# 75장 · SQLite 백업과 복구 검증

백업에서 읽은 행 수가 원본과 같을 때도 복구 결과를 다시 확인할 이유가 있습니다.

별도 DB에 백업을 복원하고 원본의 소유자·문서·본문을 다시 조회합니다. 서비스 재시작, 백업 파일 생성, 실제 데이터 복구를 각각 다른 결과로 기록합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## backup()으로 별도 데이터베이스에 복원

수업용 DB에는 alice의 과제 안내 문서 하나를 넣습니다. SQLite의 실제 백업 API로 독립된 복사본을 만듭니다. EC2 볼륨 스냅샷이나 PostgreSQL pg_dump를 실행하는 실습과 구분하세요.


### 셀 01 · SQLite 데이터베이스 백업

별개의 DB 객체를 만들고 원본에 커밋한 다음 백업합니다.


In [1]:
import sqlite3
origin = sqlite3.connect(":memory:")
origin.execute("CREATE TABLE documents(owner TEXT, id TEXT, body TEXT)")
with origin:
    origin.execute("INSERT INTO documents VALUES(?,?,?)", ("alice", "doc-1", "과제함에 제출합니다."))
backup = sqlite3.connect(":memory:")
origin.backup(backup)
print(backup.execute("SELECT count(*) FROM documents").fetchone()[0])

1


백업에서 행 하나가 보입니다. 이 단계에서는 별도 복구 대상에서 서비스가 다시 읽을 수 있는지 아직 확인하지 않았습니다.


### 백업본 한 행이 알려주는 것

<p>origin.backup(backup)은 문서 한 행을 가진 원본 DB를 별도 연결에 복사합니다. 대상에서 count(*)가 1이면 행 수를 확인한 셈입니다. 문장의 내용을 확인하려면 그 행을 직접 조회해야 합니다.</p>


## SQLite 백업 사본에서 원문까지 대조하기

API 점검에서 확인했던 답을 새 DB 연결로 복원합니다. 백업 전과 복원 후의 행 수가 같아도 본문이 다르면 복구 실패입니다.

노트북을 같은 수업 폴더에 모아 열거나 압축을 푼 장별 폴더에서 실행하세요. 시작 셀은 service-lab/probe.json을 먼저 엽니다. 파일이 없을 때만 셀에 적힌 복귀 예시를 읽습니다.

In [ ]:
from pathlib import Path
import json
here = Path.cwd()
# 각 장 폴더에서 열거나 노트북을 한 폴더에 모아 두어도 같은 실습 폴더를 씁니다.
course = here.parent if here.name in ("daily",) or (here.name.startswith("chapter") and here.name[7:].isdigit()) else here
lab = course / "service-lab"
lab.mkdir(exist_ok=True)
incoming = course / "service-lab/probe.json"
if incoming.exists():
    previous = json.loads(incoming.read_text(encoding="utf-8"))
    print("내 실습 파일", incoming.name)
else:
    previous = {'url': 'http://127.0.0.1:8080/turn',
     'expected_source': 'loan-01',
     'turn': {'id': 'loan-demo',
              'question': '대출 기간은 며칠인가요?',
              'answer': '대출 기간은 14일입니다.',
              'source': 'loan-01'}}
    print("복귀 예시", incoming.name)

내 실습 파일 probe.json


## SQLite 백업 사본에서 원문까지 대조하기 실습

앞에서 확인한 최소 예제에 내 실습 파일의 값을 넣습니다.

In [ ]:
import sqlite3
from contextlib import closing
row = previous["turn"]
db_file = lab / "turns.sqlite"
with closing(sqlite3.connect(db_file)) as saved_db, closing(sqlite3.connect(lab / "practice-backup.sqlite")) as copy:
    saved_db.execute("CREATE TABLE IF NOT EXISTS turns (id TEXT PRIMARY KEY, question TEXT, answer TEXT, source TEXT)")
    with saved_db:
        saved_db.execute("INSERT OR IGNORE INTO turns VALUES (?, ?, ?, ?)", tuple(row[k] for k in ("id","question","answer","source")))
    saved_db.backup(copy)
    saved = copy.execute("SELECT answer, source FROM turns WHERE id=?", (row["id"],)).fetchone()
result = {"id": row["id"], "expected": [row["answer"], previous["expected_source"]], "restored": list(saved)}
print(result["restored"] == result["expected"], saved[1])

True loan-01


backup이 복사한 DB에서 id로 본문과 출처를 찾아 두 필드를 함께 대조했습니다. 앞의 DB가 없으면 실습용 예시 행으로 시작합니다. 운영 DB 복원 여부는 운영 환경에서 별도로 확인합니다.

### 직접 해 보기

원본과 사본을 유지하고 기대값 사본의 출처만 바꿔 보세요. 비교 결과가 False가 되어야 합니다.

In [ ]:
# 새 리스트에 다른 출처를 넣으세요.

<details class="notebook-answer"><summary>정답 코드와 실행 결과</summary><pre><code>wrong = [row[&quot;answer&quot;], &quot;old-edition&quot;]
print(result[&quot;restored&quot;] == wrong)</code></pre><pre>False</pre><p>두 번째 값이 달라져 두 리스트가 일치하지 않습니다.</p></details>

### 76장에서 다시 열 파일

방금 계산한 값을 service-lab/restore.json에 저장합니다. 같은 장에서 조건을 바꾸고 다시 저장하면 다음 장도 그 값을 읽습니다.

In [ ]:
destination = lab / "restore.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(destination.name)

restore.json


### 셀 02 · 빈 DB 복원 후 레코드 대조

원본과 같은 조건으로 조회합니다. 새 대상에는 원래 데이터가 없었습니다.


In [2]:
target = sqlite3.connect(":memory:")
backup.backup(target)
query = "SELECT body FROM documents WHERE owner=? AND id=?"
key = ("alice", "doc-1")
original_row = origin.execute(query, key).fetchone()
restored_row = target.execute(query, key).fetchone()
print(restored_row, restored_row == original_row)

('과제함에 제출합니다.',) True


새 DB에서 읽은 문장과 비교 결과 True가 나옵니다. 조회 조건에 문서 ID뿐 아니라 소유자도 들어가므로 다른 소유자의 같은 이름을 가진 문서가 비교 대상으로 섞이지 않습니다.


#### 백업 파일이 만들어진 다음

<mark>복원 대상 DB를 별도로 열어 실제 행과 본문을 읽어야 합니다.</mark> 원본 연결로 조회하면 사본이 비어 있어도 검사가 통과할 수 있습니다. 연결한 경로를 확인한 뒤 행 수와 같은 질문의 값을 차례로 대조하세요.

선수 개념 · 70장 SQLite 문서 상태와 대화 턴 저장

### 원본·백업본·복구 대상을 나누기

<p>원본은 origin, 복사본은 backup, 복구 검사를 할 대상은 target입니다. backup.backup(target)을 호출한 뒤 target에만 변경을 가하면 원본을 비교 기준으로 계속 사용할 수 있습니다.</p>


## 행 수와 원문 값 비교

복구 대상의 문장만 다른 상황을 만들어 봅니다. 대상은 버려도 되는 수업용 메모리 DB입니다. 운영 원본을 덮어쓰지 않고도 점검 방식의 차이를 볼 수 있습니다.


### 과제 03 · 원문 필드의 불일치 검출

복원본의 doc-1만 수정한 뒤, 같은 키로 복원본에서 현재 값을 다시 읽으세요. 셀 02에서 읽어 둔 값과 새 조회값이 어떻게 다른지 출력하세요.


In [3]:
# target의 doc-1을 ‘제출 기한 미확인’으로 바꾸세요.
# 셀 02의 restored_row와 새 SELECT 결과를 나란히 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>with target:
    target.execute(&quot;UPDATE documents SET body=? WHERE owner=? AND id=?&quot;,
                   (&quot;제출 기한 미확인&quot;, *key))
current = target.execute(query, key).fetchone()
print(restored_row[0], current[0], sep=&quot; / &quot;)
print(current == original_row)</code></pre>
<p>실행 결과</p><pre><code>과제함에 제출합니다. / 제출 기한 미확인
False</code></pre>
<p>restored_row는 셀 02에서 읽어 둔 튜플입니다. DB를 바꿔도 그 튜플은 바뀌지 않습니다. 변경 뒤에 다시 SELECT한 current와 원본을 대조해야 복원본의 차이가 드러납니다.</p>
</details>


#### 행 수가 같아도 복원 검사가 실패하는 이유

```mermaid
flowchart TD
norigin["원본 DB"]
  nbackup["백업 DB"]
  ntarget["대상 DB"]
  nmatch["본문 대조"]
  nedit["대상 수정"]
  norigin --> nbackup
  nbackup --> ntarget
  ntarget --> nmatch
  nmatch --> nedit
  nedit --> nmatch
```

백업본을 별도 DB에 복원한 뒤, 대상의 본문만 바꿉니다. 각 단계에서 원본과 대상의 비교값을 읽어 보세요.

### 같은 키로 원문 비교하기

<p>restored_row와 original_row는 수정 전 값입니다. 수정한 DB에 다시 쿼리를 보내야 현재 본문을 읽을 수 있습니다. 소유자와 문서 ID를 고정하고 본문만 비교하세요.</p>


## 복원한 DB에서 빠진 문서의 소유자와 번호 찾기

행 수가 다르다는 사실만으로는 어떤 사용자의 자료가 사라졌는지 알 수 없습니다. 이번에는 서로 다른 두 사용자가 같은 문서 번호를 쓰는 상태에서 한 행의 복구가 빠진 상황을 재현합니다.

### 셀 05 · 같은 번호의 문서 두 개를 새 DB에 복원하기

이 장 셀 01에서 만든 backup을 사용합니다. 앞 절의 원본 백업을 복사한 실습 DB에 두 번째 소유자의 doc-1을 추가하세요. 같은 번호 두 개가 각각 어떤 본문인지 조회합니다.

In [4]:
import sqlite3

audit_source = sqlite3.connect(":memory:")
backup.backup(audit_source)
with audit_source:
    audit_source.execute("INSERT INTO documents VALUES (?, ?, ?)",
                         ("bob", "doc-1", "제출 기한은 금요일입니다."))
audit_target = sqlite3.connect(":memory:")
audit_source.backup(audit_target)
print(audit_target.execute("SELECT owner, id FROM documents ORDER BY owner, id").fetchall())

[('alice', 'doc-1'), ('bob', 'doc-1')]


alice와 bob 모두 doc-1을 갖고 있습니다. 단지 doc-1이 있다는 사실만 점검하면 어느 사용자의 문서가 복원됐는지 알 수 없습니다. 여기서는 실습 전용 SQLite 복사본을 사용합니다.

### 과제 06 · 빠진 행과 달라진 행을 별도로 찾기

audit_target에서 bob의 doc-1만 삭제하고 alice의 본문 한 줄은 수정합니다. 원본과 복원본을 (owner, id)로 대조해 누락된 키와 내용이 달라진 키를 각각 출력하세요. 이 셀은 실습용 복사본에서만 변경합니다.

In [5]:
# audit_target의 bob/doc-1을 삭제하고 alice/doc-1 본문을 "기한 재확인"으로 수정하세요.
# 두 DB에 대해 (owner, id)를 키로 하는 딕셔너리를 만들어 키와 본문을 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>with audit_target:
    audit_target.execute(&quot;DELETE FROM documents WHERE owner=? AND id=?&quot;, (&quot;bob&quot;, &quot;doc-1&quot;))
    audit_target.execute(&quot;UPDATE documents SET body=? WHERE owner=? AND id=?&quot;,
                         (&quot;기한 재확인&quot;, &quot;alice&quot;, &quot;doc-1&quot;))

def snapshot(con):
    rows = con.execute(&quot;SELECT owner, id, body FROM documents&quot;)
    return {(owner, doc_id): body for owner, doc_id, body in rows}

wanted, actual = snapshot(audit_source), snapshot(audit_target)
print(&quot;누락&quot;, sorted(wanted.keys() - actual.keys()))
print(&quot;내용 변경&quot;, sorted(key for key in wanted.keys() &amp; actual.keys() if wanted[key] != actual[key]))</code></pre>
<p>실행 결과</p><pre><code>누락 [(&#x27;bob&#x27;, &#x27;doc-1&#x27;)]
내용 변경 [(&#x27;alice&#x27;, &#x27;doc-1&#x27;)]</code></pre>
<p>누락 목록의 bob과 내용 변경 목록의 alice는 복구 방법이 다릅니다. 전자는 빠진 행을 찾아야 하고, 후자는 남아 있는 본문을 원본과 대조해야 합니다. 운영 PostgreSQL에서는 수업용 백업을 별도의 빈 복구 DB에 pg_restore한 뒤 동일한 조회를 수행하세요.</p>
</details>

<mark>복구 결과는 소유자와 문서 번호를 한 쌍으로 묶어 원문과 대조합니다.</mark> 문서 번호만 키로 쓰면 두 사용자의 doc-1 가운데 하나가 비교표에서 다른 하나를 덮어씁니다. 누락 판정에 쓸 키는 실제 서비스가 자료를 찾는 키와 같아야 합니다.

복구 확인서에는 확인한 키·수정 전 원문·다시 조회한 본문을 실어서 다음 담당자가 같은 행을 열 수 있게 하세요.

## 돌아보기

복원본에서 잘못된 문장이 돌아왔는데도 검사가 통과했다면, 비교에 쓴 두 값의 조회 시점을 보세요. 변경 전에 읽은 두 튜플을 다시 비교하면 변경 후 DB의 오류를 놓칩니다.
